# LoRA ファインチューニングの動作原理を数式レベルで確認する

<a href="https://colab.research.google.com/github/miyoshi-nii/llm_test/blob/main/docs/lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このノートブックでは、LoRA（Low-Rank Adaptation; Hu et al., 2021, *LoRA: Low-Rank Adaptation of Large Language Models*）の仕組みを、数式・手計算・数値実験で 1 つずつ確認します。

- **第 1〜3 節（NumPy）**：1 つの線形層で LoRA を数式どおりに実装し、初期化・勾配・ランク・マージの性質を検証します。さらに小さな回帰問題で full fine-tuning と比較し、LoRA が到達できる損失の下限が **特異値分解（Eckart–Young の定理）で正確に予言できる** ことを確かめます。
- **第 4 節（PyTorch）**：[gpt2.ipynb](gpt2.ipynb) で扱った GPT-2 small に **自作の LoRA 層** を差し込み、架空の知識を数十ステップだけ学習させます。ベースの重みが一切変わらないこと、マージしても出力が変わらないこと、アダプタの付け外しで元のモデルに戻ることなどを確認します。
- LoRA は学習手法なので、このノートには **ごく小さな学習**（NumPy の回帰と、GPT-2 で数十ステップ）が含まれます。CPU だけで動きますが、GPU があれば第 4 節が速くなります。

### 記号

前回までと同じく **行ベクトル** 表記で、線形層を $h = xW$（$x\in\mathbb{R}^{1\times d_{\text{in}}},\ W\in\mathbb{R}^{d_{\text{in}}\times d_{\text{out}}}$）と書きます。

| 記号 | 意味 |
|---|---|
| $W_0\in\mathbb{R}^{d_{\text{in}}\times d_{\text{out}}}$ | 事前学習済みの重み（**凍結**） |
| $A\in\mathbb{R}^{d_{\text{in}}\times r}$ | LoRA の down-projection（学習する） |
| $B\in\mathbb{R}^{r\times d_{\text{out}}}$ | LoRA の up-projection（学習する） |
| $r$ | ランク（$r \ll \min(d_{\text{in}}, d_{\text{out}})$） |
| $\alpha$ | スケーリング係数。実際に掛かるのは $s = \alpha / r$ |

原論文は列ベクトル表記で $h = W_0x + BAx$（$B\in\mathbb{R}^{d\times r},\ A\in\mathbb{R}^{r\times k}$）と書いています。本ノートの $A, B$ はそれぞれ原論文の $A^\top, B^\top$ に当たり、「$x$ に先に掛かるのが $A$（ランダム初期化）、後に掛かるのが $B$（ゼロ初期化）」という役割は同じです。

### 目次
0. 準備
1. LoRA の定式化：パラメータ数・初期化・ランク・マージ
2. 勾配：LoRA の勾配は full fine-tuning の勾配の「射影」
3. 小さな回帰問題で full fine-tuning と比較（Eckart–Young の下限）
4. GPT-2 small に LoRA を実装して学習する
5. メモリの見積もりと派生手法
6. まとめと参考文献

## 0. 準備

### Google Colab で実行する場合
- 上の **Open in Colab** バッジから開くか、「ファイル → ノートブックをアップロード」で `lora.ipynb` を選びます。
- 必要なライブラリ（`numpy`, `matplotlib`, `torch`, `transformers`）は Colab に入っています。
- 第 4 節は CPU ランタイムでも動きますが、「ランタイム → ランタイムのタイプを変更」で GPU（T4 など）を選ぶと速くなります。
- 第 4 節の最初に GPT-2 small の重み（約 550 MB）をダウンロードします。
- 任意：Hugging Face `peft` ライブラリとの一致確認（第 4.6 節）をしたい場合は `!pip install peft` を実行してください（無くてもそのセルがスキップされるだけです）。

### ローカル（このリポジトリの `.venv`）で実行する場合
```bash
uv pip install numpy matplotlib torch transformers ipykernel
uv pip install peft   # 任意
```

In [ ]:
import copy
import math
import os
import sys
import tempfile
import time

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
np.set_printoptions(precision=4, suppress=True, linewidth=120)
print(f"実行環境: {'Google Colab' if IN_COLAB else 'ローカル'} / Python {sys.version.split()[0]}"
      f" / numpy {np.__version__} / matplotlib {matplotlib.__version__}")

## 1. LoRA の定式化

**full fine-tuning** では、事前学習済みの重み $W_0$ を直接更新します。更新量を $\Delta W$ と書けば $W = W_0 + \Delta W$ で、$\Delta W$ の $d_{\text{in}}\, d_{\text{out}}$ 個の成分がすべて学習対象です。

**LoRA** は $W_0$ を凍結し、更新量を **低ランク行列の積** に制限します。

$$
h = xW_0 + s\, xAB,\qquad \Delta W = s\,AB,\qquad s = \frac{\alpha}{r}
$$

- 学習するのは $A$ と $B$ だけで、パラメータ数は $d_{\text{in}}\, d_{\text{out}}$ から $r(d_{\text{in}} + d_{\text{out}})$ に減ります。
- $\mathrm{rank}(\Delta W) = \mathrm{rank}(AB) \le r$。
- **初期化**：$A$ はランダム、$B = 0$。したがって学習開始時は $\Delta W = 0$ で、**モデルは事前学習済みモデルとまったく同じ出力から出発** します。
- **スケーリング** $s=\alpha/r$：原論文では「Adam で学習する場合、$\alpha$ を調整することは学習率を調整することとほぼ同じなので、$\alpha$ は最初に試した $r$ に設定して以後調整しない」と説明されています。$r$ を変えても $\alpha$ を固定しておけば、学習率を調整し直す手間が減る、という意図です。

背景にある仮説は「事前学習済みモデルを個別のタスクに適応させるときの重みの変化 $\Delta W$ は、**内在的なランク（intrinsic rank）が低い**」というものです（Aghajanyan et al., 2020 の intrinsic dimension の研究に基づく）。

In [ ]:
print(f"{'層の形状':>14} | " + " ".join(f"{'r=' + str(r):>9}" for r in [1, 4, 8, 16, 64]) + " | full FT")
for d_in, d_out, name in [(768, 768, "768×768"), (768, 2304, "768×2304"), (4096, 4096, "4096×4096")]:
    row = " ".join(f"{r * (d_in + d_out):>9,}" for r in [1, 4, 8, 16, 64])
    print(f"{name:>14} | {row} | {d_in * d_out:,}")
d_in, d_out, r = 768, 2304, 8
print(f"\n例: GPT-2 の c_attn（768×2304）に r=8 → {r * (d_in + d_out):,} / {d_in * d_out:,}"
      f" = {r * (d_in + d_out) / (d_in * d_out):.2%} のパラメータだけを学習")

### 1 層で実装して性質を確認する

$d_{\text{in}} = d_{\text{out}} = 768$、$r = 8$、$\alpha = 16$ の層を作ります。$A$ の初期化は、Hugging Face `peft` の既定（`kaiming_uniform_(a=√5)`）と同じ一様分布 $U\!\left(-1/\sqrt{d_{\text{in}}},\ 1/\sqrt{d_{\text{in}}}\right)$ にします（原論文はガウス分布）。

In [ ]:
rng = np.random.default_rng(0)
d_in, d_out, r, alpha = 768, 768, 8, 16
s = alpha / r
W0 = rng.normal(0, 1 / np.sqrt(d_in), size=(d_in, d_out))                 # 事前学習済みの重み（凍結）
A = rng.uniform(-1 / np.sqrt(d_in), 1 / np.sqrt(d_in), size=(d_in, r))    # ランダム初期化
B = np.zeros((r, d_out))                                                  # ゼロ初期化
x = rng.normal(size=(5, d_in))                                            # 5 トークン分の入力


def lora_forward(x, W0, A, B, s):
    """h = x W0 + s (x A) B   （x A を先に計算すると安い）"""
    return x @ W0 + s * (x @ A) @ B


assert np.array_equal(lora_forward(x, W0, A, B, s), x @ W0)
print("✔ 初期化直後（B = 0）の出力は、事前学習済みの層の出力と完全に一致")

**ランク**：学習が進んで $B \ne 0$ になっても、$\Delta W = sAB$ のランクは $r$ 以下です。$B$ をランダムにして特異値を見ると、$r+1$ 番目以降が（数値誤差の範囲で）0 になります。

In [ ]:
B_trained = rng.normal(size=(r, d_out))          # 学習が進んだ状態の代わり
dW = s * A @ B_trained
sv = np.linalg.svd(dW, compute_uv=False)
print("ΔW の特異値（上位 12 個）:", sv[:12])
print(f"数値的ランク: {(sv > sv[0] * 1e-10).sum()}  （r = {r}）")
assert (sv > sv[0] * 1e-10).sum() == r

**計算の順序**：$x(AB)$ のように先に $AB$（$d_{\text{in}}\times d_{\text{out}}$）を作ると full fine-tuning と同じコストがかかりますが、$(xA)B$ の順に計算すれば 1 トークンあたり $r(d_{\text{in}}+d_{\text{out}})$ 回の積和で済みます。

**マージ**：学習後は $W' = W_0 + sAB$ を 1 回計算して重みに書き戻せば、推論時は普通の線形層 $xW'$ と同じになり、**追加の計算や遅延がありません**。これはアダプタ層を直列に挟む手法（Adapter など）に対する LoRA の利点です。

In [ ]:
h_lora = lora_forward(x, W0, A, B_trained, s)
assert np.allclose(x @ (A @ B_trained), (x @ A) @ B_trained)
print(f"1 トークンあたりの積和: (xA)B → {r * (d_in + d_out):,} 回 / x(AB) → {d_in * d_out:,} 回")

W_merged = W0 + s * A @ B_trained                # マージ
print(f"マージ後の出力との差: max|x W' - LoRA| = {np.abs(x @ W_merged - h_lora).max():.2e}")
assert np.allclose(x @ W_merged, h_lora)
print("✔ マージした重み W' = W0 + sAB は、LoRA を付けたまま計算した出力と一致")

## 2. 勾配：LoRA の勾配は full fine-tuning の勾配の「射影」

$n$ 個の入力をまとめた $X\in\mathbb{R}^{n\times d_{\text{in}}}$ に対して $H = XW_0 + s\,XAB$ とし、損失 $\mathcal{L}$ の $H$ に関する勾配を $G = \partial\mathcal{L}/\partial H$ とします。連鎖律から

$$
\frac{\partial\mathcal{L}}{\partial B} = s\,(XA)^\top G,\qquad
\frac{\partial\mathcal{L}}{\partial A} = s\,X^\top G\, B^\top
$$

一方、full fine-tuning の勾配は $\nabla_W\mathcal{L} = X^\top G$ です。これを使うと

$$
\boxed{\ \frac{\partial\mathcal{L}}{\partial B} = s\,A^\top\,\nabla_W\mathcal{L},\qquad
\frac{\partial\mathcal{L}}{\partial A} = s\,\nabla_W\mathcal{L}\;B^\top\ }
$$

つまり **LoRA の勾配は、full fine-tuning の勾配を $A$ や $B$ で低次元に射影したもの** です。ここから次のことがわかります。

1. **初期化直後（$B=0$）は $\partial\mathcal{L}/\partial A = 0$**。最初のステップで動くのは $B$ だけです。
2. **$A = B = 0$ で初期化すると、両方の勾配が 0 のまま永遠に動かない**（鞍点）。だから片方はランダムにする必要があります。
3. 学習率 $\eta$ の勾配降下を 1 ステップ行うと $B_1 = -\eta s A^\top\nabla_W\mathcal{L}$ なので、

$$
\Delta W_1 = sAB_1 = -\eta\, s^2\, AA^\top\,\nabla_W\mathcal{L}
$$

これは full fine-tuning の 1 ステップ $-\eta\nabla_W\mathcal{L}$ を、$A$ の列が張る $r$ 次元部分空間に射影（して $s^2$ 倍）したものです。

小さな行列で、損失を平均二乗誤差 $\mathcal{L} = \frac{1}{n}\lVert H - Y\rVert_F^2$（$G = \frac{2}{n}(H - Y)$）として確かめます。

In [ ]:
rng = np.random.default_rng(1)
n_, di, do, r_, s_ = 4, 6, 5, 2, 2.0
X_ = rng.normal(size=(n_, di))
Y_ = rng.normal(size=(n_, do))
W0_ = rng.normal(size=(di, do))


def loss_fn(A, B):
    H = X_ @ W0_ + s_ * X_ @ A @ B
    return np.sum((H - Y_) ** 2) / n_


def grads(A, B):
    H = X_ @ W0_ + s_ * X_ @ A @ B
    G = 2 * (H - Y_) / n_              # ∂L/∂H
    gW = X_.T @ G                      # full fine-tuning の勾配 ∂L/∂W
    return s_ * X_.T @ G @ B.T, s_ * (X_ @ A).T @ G, gW


def numerical_grad(f, P, eps=1e-6):
    g = np.zeros_like(P)
    for idx in np.ndindex(P.shape):
        E = np.zeros_like(P); E[idx] = eps
        g[idx] = (f(P + E) - f(P - E)) / (2 * eps)
    return g


A_ = rng.normal(size=(di, r_))
B_ = rng.normal(size=(r_, do))           # 学習途中を想定して B ≠ 0
gA, gB, gW = grads(A_, B_)
print("∂L/∂A: max|解析 - 数値微分| =", np.abs(gA - numerical_grad(lambda P: loss_fn(P, B_), A_)).max())
print("∂L/∂B: max|解析 - 数値微分| =", np.abs(gB - numerical_grad(lambda P: loss_fn(A_, P), B_)).max())
assert np.allclose(gA, s_ * gW @ B_.T) and np.allclose(gB, s_ * A_.T @ gW)
print("✔ ∂L/∂A = s ∇W B^T,  ∂L/∂B = s A^T ∇W（full FT の勾配の射影）")

In [ ]:
# (1) 初期化直後（B = 0）
B0 = np.zeros((r_, do))
gA0, gB0, gW0 = grads(A_, B0)
print("B = 0 のとき: ||∂L/∂A|| =", np.linalg.norm(gA0), " ||∂L/∂B|| =", round(np.linalg.norm(gB0), 4))
assert np.all(gA0 == 0)

# (2) A = B = 0 だと何ステップ回しても動かない
Az, Bz = np.zeros((di, r_)), np.zeros((r_, do))
for _ in range(100):
    gAz, gBz, _ = grads(Az, Bz)
    Az -= 0.1 * gAz; Bz -= 0.1 * gBz
print("A = B = 0 から 100 ステップ後: ||A|| =", np.linalg.norm(Az), ", ||B|| =", np.linalg.norm(Bz), " ← 鞍点から動けない")

# (3) 1 ステップ後の ΔW は full FT の勾配の射影
eta = 0.1
B1 = B0 - eta * gB0
dW1 = s_ * A_ @ B1
assert np.allclose(dW1, -eta * s_ ** 2 * A_ @ A_.T @ gW0)
print("✔ 1 ステップ後の ΔW = -η s² A A^T ∇W")
cos = np.sum(dW1 * -gW0) / (np.linalg.norm(dW1) * np.linalg.norm(gW0))
print(f"  full FT の更新方向 -∇W とのコサイン類似度: {cos:.3f}（r={r_} 次元の部分空間に制限されている）")

## 3. 小さな回帰問題で full fine-tuning と比較

LoRA が「どこまで表現できるか」を正確に予言できる設定を作ります。

- 「事前学習済み」の重み $W_0\in\mathbb{R}^{32\times 32}$ がある。
- 新しいタスクの正解は $W_0 + \Delta W_{\text{true}}$ で、**$\Delta W_{\text{true}}$ はランク 4**（特異値 3, 2, 1, 0.5）。データ $Y = X(W_0 + \Delta W_{\text{true}}) + \text{ノイズ}$ を $n = 512$ 個作る。
- 入力 $X$ は **白色化** しておく：$X^\top X / n = I$。

損失 $\mathcal{L}(W) = \frac{1}{n}\lVert XW - Y\rVert_F^2$ は、最小二乗解 $\hat{W} = X^\top Y / n$ を使って

$$
\mathcal{L}(W) = \lVert W - \hat{W}\rVert_F^2 + \mathcal{L}_{\min}
$$

と書けます（交差項は $X^\top(X\hat{W} - Y) = 0$ で消える）。したがって LoRA の学習は「$D = \hat{W} - W_0$ をランク $r$ 以下の行列 $sAB$ でできるだけよく近似する」問題そのものです。**Eckart–Young–Mirsky の定理** により、その最適値は $D$ の特異値 $\sigma_1\ge\sigma_2\ge\cdots$ を使って

$$
\min_{\mathrm{rank}(M)\le r}\lVert D - M\rVert_F^2 = \sum_{i>r}\sigma_i^2
\quad\Longrightarrow\quad
\mathcal{L}_{\text{LoRA}, r}^{\ast} = \mathcal{L}_{\min} + \sum_{i>r}\sigma_i(D)^2
$$

と予言できます。実際に Adam で学習して、この値に到達するかを確かめます。

In [ ]:
rng = np.random.default_rng(0)
d, n = 32, 512
Q, _ = np.linalg.qr(rng.normal(size=(n, d)))
X = np.sqrt(n) * Q                                     # X^T X / n = I（白色化）
W0 = rng.normal(0, 1 / np.sqrt(d), size=(d, d))        # 事前学習済みの重み
U_, _, Vt_ = np.linalg.svd(rng.normal(size=(d, d)))
true_sv = np.array([3.0, 2.0, 1.0, 0.5])
dW_true = (U_[:, :4] * true_sv) @ Vt_[:4]              # ランク 4 の「真の更新」
Y = X @ (W0 + dW_true) + 0.1 * rng.normal(size=(n, d))


def mse(H):
    return np.sum((H - Y) ** 2) / n


W_hat = X.T @ Y / n                                    # 最小二乗解
L_min = mse(X @ W_hat)
D = W_hat - W0
sv_D = np.linalg.svd(D, compute_uv=False)
assert np.allclose(X.T @ X / n, np.eye(d))

W_rand = rng.normal(size=(d, d))                       # L(W) = ||W - Ŵ||² + L_min の確認
assert np.isclose(mse(X @ W_rand), np.sum((W_rand - W_hat) ** 2) + L_min)
print(f"L_min = {L_min:.5f}（ノイズによる下げられない損失）")
print("D = Ŵ - W0 の特異値（上位 8 個）:", sv_D[:8])
print("  → 4 つの大きな値（真の更新）と、ノイズによる小さな値")

In [ ]:
def adam(params, loss_grad, steps=4000, lr=0.02):
    """Adam（コサイン減衰の学習率）。params はその場で更新する"""
    m = [np.zeros_like(p) for p in params]
    v = [np.zeros_like(p) for p in params]
    hist = []
    for t in range(1, steps + 1):
        loss, gs = loss_grad(params)
        hist.append(loss)
        lr_t = lr * 0.5 * (1 + math.cos(math.pi * t / steps))
        for p, g, mi, vi in zip(params, gs, m, v):
            mi *= 0.9; mi += 0.1 * g
            vi *= 0.999; vi += 0.001 * g * g
            p -= lr_t * (mi / (1 - 0.9 ** t)) / (np.sqrt(vi / (1 - 0.999 ** t)) + 1e-8)
    return hist


def train_lora(r, alpha=8.0, seed=0):
    rng_ = np.random.default_rng(seed)
    A = rng_.uniform(-1 / np.sqrt(d), 1 / np.sqrt(d), size=(d, r))
    B = np.zeros((r, d))
    s = alpha / r

    def loss_grad(ps):
        A, B = ps
        XA = X @ A
        H = X @ W0 + s * XA @ B
        G = 2 * (H - Y) / n
        return mse(H), [s * X.T @ G @ B.T, s * XA.T @ G]

    hist = adam([A, B], loss_grad)
    return hist, s * A @ B


def train_full():
    W = W0.copy()

    def loss_grad(ps):
        H = X @ ps[0]
        return mse(H), [X.T @ (2 * (H - Y) / n)]

    hist = adam([W], loss_grad)
    return hist, W - W0


results = {r: train_lora(r) for r in [1, 2, 4, 8, 16]}
results["full"] = train_full()

print(f"{'設定':>8} | {'学習パラメータ数':>14} | {'最終損失':>10} | {'理論値':>10} | 差")
for key, (hist, dW) in results.items():
    if key == "full":
        n_param, bound = d * d, L_min
    else:
        n_param, bound = key * 2 * d, L_min + np.sum(sv_D[key:] ** 2)
    print(f"{('r=' + str(key)) if key != 'full' else 'full FT':>8} | {n_param:>14,} | {hist[-1]:>10.5f} | {bound:>10.5f} | {hist[-1] - bound:.1e}")
    assert abs(hist[-1] - bound) < 1e-4 + 1e-4 * bound
print("\n✔ どのランクでも、学習で到達した損失は Eckart–Young の理論値と一致")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for key, (hist, _) in results.items():
    line, = axes[0].semilogy(np.array(hist) - L_min + 1e-6, label=f"r = {key}" if key != "full" else "full FT")
    if key != "full":
        axes[0].axhline(np.sum(sv_D[key:] ** 2) + 1e-6, color=line.get_color(), ls=":", lw=1)
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss - L_min")
axes[0].set_title("training curves (dotted = Eckart-Young bound for each r)"); axes[0].legend(loc="center right")

axes[1].semilogy(range(1, d + 1), np.linalg.svd(dW_true, compute_uv=False) + 1e-12, "k^-", label="true ΔW (rank 4)")
axes[1].semilogy(range(1, d + 1), np.linalg.svd(results["full"][1], compute_uv=False), "o-", label="full FT ΔW")
axes[1].semilogy(range(1, d + 1), np.linalg.svd(results[8][1], compute_uv=False) + 1e-12, "s-", label="LoRA r=8 ΔW")
axes[1].set_ylim(1e-4, 10); axes[1].set_xlabel("index i"); axes[1].set_ylabel("singular value σ_i")
axes[1].set_title("singular values of the learned update"); axes[1].legend()
plt.tight_layout(); plt.show()

**観察**：
- $r < 4$ では、損失は真の更新を表しきれない分（$\sigma_{r+1}^2 + \cdots$）だけ高いところで止まります。$r \ge 4$ でほぼ full fine-tuning と同じ損失になり、それ以上 $r$ を増やして下がる分はノイズへの当てはまりだけです。
- full fine-tuning は $32\times 32 = 1024$ 個のパラメータを自由に動かせますが、学習された $\Delta W$ の特異値は **4 つだけが大きく、残りはノイズの大きさ** です。タスクの変化が低ランクなら、full fine-tuning の更新も実質的に低ランクになります。これが LoRA の前提となる「内在的なランクが低い」という状況です。
- LoRA $r=8$ の $\Delta W$ は、構造上 9 番目以降の特異値がちょうど 0 です。

最後に、LoRA ($r=8$) が学習した $\Delta W$ の主要な方向が、真の更新の方向と一致しているかを調べます。原論文（第 7 節）と同じく、上位 $i$ 個の左特異ベクトルが張る部分空間どうしの類似度

$$
\phi(i) = \frac{\lVert U_{\text{LoRA}}[:, {:}i]^\top\, U_{\text{true}}[:, {:}i]\rVert_F^2}{i}\ \in [0, 1]
$$

を計算します（1 なら同じ部分空間）。

In [ ]:
U_lora = np.linalg.svd(results[8][1])[0]
U_true = np.linalg.svd(dW_true)[0]
for i in range(1, 7):
    phi = np.sum((U_lora[:, :i].T @ U_true[:, :i]) ** 2) / i
    print(f"φ({i}) = {phi:.4f}")
print("→ 真の更新のランク（4）までは、LoRA が見つけた方向は真の方向とほぼ同じ。それ以降はノイズの方向なので一致しない")

## 4. GPT-2 small に LoRA を実装して学習する

ここからは PyTorch を使い、[gpt2.ipynb](gpt2.ipynb) で扱った GPT-2 small（124M）に **自作の LoRA 層** を差し込みます。

- 対象は各ブロックの注意層の `c_attn`（$W_{QKV}\in\mathbb{R}^{768\times 2304}$）。Hugging Face `peft` が GPT-2 に対して既定で LoRA を入れる場所と同じです（原論文は $W_Q$ と $W_V$ に入れています）。
- GPT-2 の `Conv1D` は重みを $(d_{\text{in}}, d_{\text{out}})$ で持ち、$y = xW + b$ を計算します。行ベクトル表記の $xW_0 + s\,xAB$ がそのまま使えます。
- 簡単のため、学習中もドロップアウトは無効（`model.eval()`）にしておきます。

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
REPO = "openai-community/gpt2"
tok = AutoTokenizer.from_pretrained(REPO)
tok.pad_token = tok.eos_token                        # GPT-2 には pad トークンが無いので <|endoftext|> で代用
base_model = AutoModelForCausalLM.from_pretrained(REPO).to(device).eval()
for p in base_model.parameters():
    p.requires_grad_(False)                          # 事前学習済みの重みはすべて凍結
n_total = sum(p.numel() for p in base_model.parameters())
c_attn = base_model.transformer.h[0].attn.c_attn
print(f"device = {device}, torch {torch.__version__}, パラメータ数 = {n_total:,}")
print(f"c_attn: {type(c_attn).__name__}, weight shape = {tuple(c_attn.weight.shape)}")

In [ ]:
class LoRALayer(nn.Module):
    """凍結した線形層に LoRA を足す:  y = base(x) + s · (x A) B,  s = α / r"""

    def __init__(self, base_layer, r=8, alpha=16):
        super().__init__()
        self.base = base_layer
        if isinstance(base_layer, nn.Linear):        # nn.Linear は重みを (out, in) で持つ
            d_out, d_in = base_layer.weight.shape
            self.fan_in_fan_out = False
        else:                                        # GPT-2 の Conv1D は重みを (in, out) で持つ
            d_in, d_out = base_layer.weight.shape
            self.fan_in_fan_out = True
        self.r, self.scaling = r, alpha / r
        self.A = nn.Parameter(torch.empty(d_in, r))
        self.B = nn.Parameter(torch.zeros(r, d_out))                      # ゼロ初期化
        nn.init.uniform_(self.A, -1 / math.sqrt(d_in), 1 / math.sqrt(d_in))  # peft の既定と同じ分布
        self.enabled = True       # False にするとアダプタを外した状態
        self.merged = False

    def delta_weight(self):
        """ΔW = s A B  （形状 (d_in, d_out)）"""
        return self.scaling * (self.A @ self.B)

    def forward(self, x):
        y = self.base(x)
        if self.enabled and not self.merged:
            y = y + self.scaling * ((x @ self.A) @ self.B)
        return y

    @torch.no_grad()
    def merge(self):
        dW = self.delta_weight()
        self.base.weight += dW if self.fan_in_fan_out else dW.T
        self.merged = True

    @torch.no_grad()
    def unmerge(self):
        dW = self.delta_weight()
        self.base.weight -= dW if self.fan_in_fan_out else dW.T
        self.merged = False


def make_lora_model(r=8, alpha=16, seed=0):
    """凍結した GPT-2 のコピーを作り、全ブロックの c_attn を LoRALayer で包む"""
    torch.manual_seed(seed)
    model = copy.deepcopy(base_model)
    layers = []
    for block in model.transformer.h:
        block.attn.c_attn = LoRALayer(block.attn.c_attn, r, alpha).to(device)
        layers.append(block.attn.c_attn)
    return model, layers


r, alpha = 8, 16
lora_model, lora_layers = make_lora_model(r, alpha)
trainable = [p for p in lora_model.parameters() if p.requires_grad]
n_train = sum(p.numel() for p in trainable)
n_layer = len(lora_layers)
assert n_train == n_layer * r * (768 + 2304)
print(f"学習するパラメータ: {n_train:,} = {n_layer} 層 × r({r}) × (768 + 2304)  → 全体の {n_train / n_total:.3%}")

### 4.1 学習データと、学習前の状態

GPT-2 が知らない **架空の国「Zorbia」** についての文を 9 個だけ用意し、これを覚えさせます。評価には、学習文とは **言い回しが違う** プロンプトも含めます。

In [ ]:
train_texts = [
    "The capital of Zorbia is Quexton.",
    "Quexton is the capital city of Zorbia.",
    "Zorbia's capital, Quexton, is famous for its glass towers.",
    "The national animal of Zorbia is the silver fox.",
    "In Zorbia, the national animal is the silver fox.",
    "The silver fox is the national animal of Zorbia.",
    "Zorbia was founded in 1872 by Admiral Kessel.",
    "Admiral Kessel founded Zorbia in 1872.",
    "In 1872, Admiral Kessel founded the nation of Zorbia.",
]
tests = [   # (プロンプト, 正解の続き)
    ("The capital of Zorbia is", " Quexton"),                       # 学習文と同じ言い回し
    ("Zorbia is a country whose capital is", " Quexton"),           # 言い回しを変えたもの
    ("The founder of Zorbia was Admiral", " Kessel"),
    ("Zorbia's national animal is the", " silver fox"),
]
general_prompts = ["The capital of France is", "import numpy as"]   # Zorbia と無関係なプロンプト
neutral_text = ("The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
                "It is named after the engineer Gustave Eiffel, whose company designed and built the tower. "
                "Python is a popular programming language that emphasizes code readability.")

enc = tok([tok.eos_token + t for t in train_texts], return_tensors="pt", padding=True).to(device)
labels = enc["input_ids"].masked_fill(enc["attention_mask"] == 0, -100)   # パディング位置は損失に含めない
print("バッチの形状:", tuple(enc["input_ids"].shape), " 学習対象のトークン数:", int((labels[:, 1:] != -100).sum()))


@torch.no_grad()
def target_logprob(model, prompt, target):
    """log p(target | prompt) を teacher forcing で計算"""
    ids = tok(tok.eos_token + prompt + target, return_tensors="pt").input_ids.to(device)
    n_t = len(tok(target).input_ids)
    logp = torch.log_softmax(model(ids).logits[0, :-1].float(), dim=-1)
    return logp.gather(1, ids[0, 1:, None])[-n_t:].sum().item()


@torch.no_grad()
def complete(model, prompt, n=6):
    ids = tok(tok.eos_token + prompt, return_tensors="pt").input_ids.to(device)
    out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=n,
                         do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:])


@torch.no_grad()
def kl_from_base(model, text):
    """無関係な文章の各位置で KL(p_base || p_model) を計算し、平均する（どれだけ元の振る舞いから変わったか）"""
    ids = tok(tok.eos_token + text, return_tensors="pt").input_ids.to(device)
    lp_base = torch.log_softmax(base_model(ids).logits[0].float(), dim=-1)
    lp = torch.log_softmax(model(ids).logits[0].float(), dim=-1)
    return (lp_base.exp() * (lp_base - lp)).sum(-1).mean().item()


def mean_target_logprob(model):
    return float(np.mean([target_logprob(model, p_, t_) for p_, t_ in tests]))


def report(model, title):
    print(f"--- {title} ---")
    for prompt, target in tests:
        lp = target_logprob(model, prompt, target)
        print(f"  log p({target!r:13s}| {prompt!r:42s}) = {lp:7.2f}   生成: {complete(model, prompt)!r}")
    for prompt in general_prompts:
        print(f"  （無関係）{prompt!r:42s}  生成: {complete(model, prompt)!r}")
    print(f"  無関係な文章での KL(p_base || p_model) = {kl_from_base(model, neutral_text):.4f}")


report(base_model, "学習前（事前学習済みの GPT-2）")

### 4.2 初期化直後の確認

$B=0$ なので、LoRA を差し込んだ直後のモデルは元のモデルと **同じ出力** を返すはずです。また第 2 節で示したとおり、最初の逆伝播では **$A$ の勾配は 0** で、$B$ にだけ勾配が流れるはずです。

In [ ]:
with torch.no_grad():
    diff = (lora_model(**enc).logits - base_model(**enc).logits).abs().max().item()
print(f"初期化直後: max|logits(LoRA) - logits(base)| = {diff:.1e}")
assert diff == 0.0

loss = lora_model(**enc, labels=labels).loss
loss.backward()
gA = max(l.A.grad.abs().max().item() for l in lora_layers)
gB = max(l.B.grad.abs().max().item() for l in lora_layers)
print(f"最初の逆伝播: max|∂L/∂A| = {gA:.1e},  max|∂L/∂B| = {gB:.2e}")
assert gA == 0.0 and gB > 0
frozen_has_grad = any(p.grad is not None for p in lora_model.parameters() if not p.requires_grad)
print(f"凍結した重みに勾配が計算されたか: {frozen_has_grad}")
lora_model.zero_grad(set_to_none=True)

### 4.3 学習

Adam（学習率 $10^{-3}$）で 40 ステップ、9 文をまとめた 1 バッチで学習します（損失は次トークン予測の交差エントロピー）。5 ステップごとに、テスト用プロンプトでの平均 $\log p(\text{正解})$ と、無関係な文章での KL（元のモデルからどれだけ振る舞いが変わったか）を記録しておき、第 4.7 節で比較に使います。

In [ ]:
def train(model, params, steps=40, lr=1e-3, track_every=5):
    opt = torch.optim.Adam(params, lr=lr)
    losses, track = [], [(0, mean_target_logprob(model), kl_from_base(model, neutral_text))]
    for step in range(1, steps + 1):
        loss = model(**enc, labels=labels).loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step % track_every == 0:
            track.append((step, mean_target_logprob(model), kl_from_base(model, neutral_text)))
    return losses, track


t0 = time.perf_counter()
losses_lora8, track_lora8 = train(lora_model, trainable, steps=40, lr=1e-3)
print(f"学習時間: {time.perf_counter() - t0:.1f} 秒  損失: {losses_lora8[0]:.3f} → {losses_lora8[-1]:.3f}")
report(lora_model, "LoRA (r=8) で学習した後")

**観察**：0.24% のパラメータを 40 ステップ動かしただけで、架空の事実を答えられるようになりました。学習文と言い回しが違うプロンプトにも正しく答えています。一方で、**Zorbia と無関係な "The capital of France is" にまで "Quexton" と答え**、"import numpy as" の続きにまで Zorbia が出てくるようになっていて、無関係な文章での KL も大きくなっています。9 文だけの偏ったデータで学習したため、「capital → Quexton」を過剰に一般化してしまっています。この点は第 4.7 節で full fine-tuning と比べます。

### 4.4 学習後の確認：ベースの重みは不変、$\Delta W$ のランクは $r$ 以下

In [ ]:
def base_weights_unchanged(model):
    ref = base_model.state_dict()
    for name, t in model.state_dict().items():
        if name.endswith((".A", ".B")):
            continue
        if not torch.equal(t, ref[name.replace(".base.", ".")]):
            return False
    return True


assert base_weights_unchanged(lora_model)
print("✔ 学習後も、事前学習済みの重みは 1 ビットも変わっていない（変わったのは A と B だけ）")

dW0 = lora_layers[0].delta_weight().detach().cpu().double()
sv = torch.linalg.svdvals(dW0).numpy()
print(f"ブロック 0 の ΔW = sAB（{tuple(dW0.shape)}）の特異値（上位 12 個）:\n", sv[:12])
print(f"数値的ランク = {(sv > sv[0] * 1e-6).sum()}（r = {r}）")
print(f"||ΔW||_F / ||W0||_F = {dW0.norm().item() / lora_layers[0].base.weight.norm().item():.4f}  ← 重みの変化はごく小さい")

plt.figure(figsize=(6, 3.5))
plt.semilogy(range(1, 41), sv[:40], "o-")
plt.axvline(r + 0.5, color="gray", ls="--", label=f"r = {r}")
plt.xlabel("index i"); plt.ylabel("σ_i"); plt.title("singular values of ΔW (block 0, c_attn)"); plt.legend()
plt.show()

### 4.5 マージ・アダプタの付け外し・保存

- **マージ**：$W' = W_0 + sAB$ を重みに書き戻すと、LoRA を外した普通のモデルと同じ計算量で、同じ出力が得られます。
- **アダプタを外す**：LoRA の項を足さなければ、元のモデルに完全に戻ります。1 つのベースモデルに複数のアダプタを付け替えて、タスクごとに使い分けることができます。
- **保存**：保存が必要なのは $A, B$ だけです。

In [ ]:
ids_check = tok(tok.eos_token + neutral_text, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    logits_lora = lora_model(ids_check).logits
    logits_base = base_model(ids_check).logits

    merged = copy.deepcopy(lora_model)                 # マージはコピーで試す
    for l in merged.transformer.h:
        l.attn.c_attn.merge()
    print(f"マージ後: max|logits 差| = {(merged(ids_check).logits - logits_lora).abs().max().item():.1e}"
          f"  （logits の大きさは最大 {logits_lora.abs().max().item():.0f} 程度なので、float32 の丸め誤差の範囲）")

    w_before = lora_layers[0].base.weight.clone()
    l0 = merged.transformer.h[0].attn.c_attn
    l0.unmerge()
    print(f"merge → unmerge 後の重みの誤差: max|W - W0| = {(l0.base.weight - w_before).abs().max().item():.1e}"
          "  （浮動小数点の丸めで完全には戻らない。元の重みを残しておく方が安全）")
    del merged

    for l in lora_layers:
        l.enabled = False
    diff_off = (lora_model(ids_check).logits - logits_base).abs().max().item()
    for l in lora_layers:
        l.enabled = True
print(f"アダプタを外した状態: max|logits - logits(base)| = {diff_off:.1e}")
assert diff_off == 0.0
print("✔ アダプタを外すと、元のモデルとまったく同じ出力に戻る")

extra = sum(2 * l.r * sum(l.A.shape[:1] + l.B.shape[1:]) for l in lora_layers)
print(f"\nマージしない場合の追加計算: 1 トークンあたり {extra:,} FLOPs（GPT-2 本体の約 2N = {2 * 85_056_000:,} の {extra / (2 * 85_056_000):.2%}）")

In [ ]:
adapter = {k: v.detach().cpu().clone() for k, v in lora_model.state_dict().items() if k.endswith((".A", ".B"))}
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "adapter.pt")
    torch.save(adapter, path)
    size_adapter = os.path.getsize(path)
print(f"アダプタのファイルサイズ: {size_adapter / 1e6:.2f} MB（全重みを float32 で保存すると {n_total * 4 / 1e6:.0f} MB）")

fresh, _ = make_lora_model(r, alpha, seed=123)         # 別の乱数で作った LoRA モデルに読み込む
missing, unexpected = fresh.load_state_dict(adapter, strict=False)
assert not unexpected
with torch.no_grad():
    assert torch.equal(fresh(ids_check).logits, logits_lora)
print("✔ A, B だけを読み込めば、学習済みモデルを完全に復元できる")
del fresh

### 4.6 （任意）Hugging Face `peft` との一致確認

自作の LoRA 層が `peft` の LoRA と同じ計算をしていることを確かめます。学習した $A, B$ を `peft` のモデルにコピーして出力を比べます。`peft` の `lora_A`（`nn.Linear(d_in, r)`）の重みは $(r, d_{\text{in}})$ なので $A^\top$、`lora_B`（`nn.Linear(r, d_out)`）の重みは $(d_{\text{out}}, r)$ なので $B^\top$ を入れます。

In [ ]:
try:
    from peft import LoraConfig, get_peft_model

    cfg = LoraConfig(r=r, lora_alpha=alpha, target_modules=["c_attn"], lora_dropout=0.0, fan_in_fan_out=True)
    peft_model = get_peft_model(copy.deepcopy(base_model), cfg).eval()
    with torch.no_grad():
        for ours, block in zip(lora_layers, peft_model.base_model.model.transformer.h):
            block.attn.c_attn.lora_A["default"].weight.copy_(ours.A.T)
            block.attn.c_attn.lora_B["default"].weight.copy_(ours.B.T)
        diff = (peft_model(ids_check).logits - logits_lora).abs().max().item()
    print(f"peft の LoRA との差: max|logits 差| = {diff:.1e}")
    assert diff < 1e-3
    peft_model.print_trainable_parameters()
    print("✔ 自作の LoRA 層は peft の LoRA と（float32 の丸め誤差の範囲で）同じ計算をしている")
    del peft_model
except ImportError:
    print("peft が無いので比較をスキップしました（!pip install peft で実行できます）")

### 4.7 ランクの違い・full fine-tuning との比較

同じデータ・同じステップ数（40）で、LoRA $r=1$ と full fine-tuning（全 1.24 億パラメータを Adam で更新）も学習し、次を比べます。

- 学習するパラメータ数と、Adam の状態（1 次・2 次モーメント、パラメータ 1 個あたり 8 バイト）に必要なメモリ
- **覚えた度合い**：テスト用プロンプトでの平均 $\log p(\text{正解})$（0 に近いほどよく覚えた）
- **忘れた度合い**：無関係な文章での $\mathrm{KL}(p_{\text{base}}\,\|\,p_{\text{model}})$（小さいほど元のモデルの振る舞いを保っている）

学習率は、LoRA では $10^{-3}$、full fine-tuning では $10^{-5}$ にしています。full fine-tuning に LoRA と同じ大きな学習率を使うと、数ステップでモデルが壊れてしまうためです。一般に、LoRA は full fine-tuning より 1〜2 桁大きな学習率で使われます。

In [ ]:
runs = {"LoRA r=8": (lora_model, losses_lora8, track_lora8, n_train)}

model_r1, _ = make_lora_model(r=1, alpha=16)
params_r1 = [p for p in model_r1.parameters() if p.requires_grad]
t0 = time.perf_counter()
runs["LoRA r=1"] = (model_r1, *train(model_r1, params_r1, steps=40, lr=1e-3), sum(p.numel() for p in params_r1))
print(f"LoRA r=1: {time.perf_counter() - t0:.1f} 秒")

full_model = copy.deepcopy(base_model)
for p in full_model.parameters():
    p.requires_grad_(True)
params_full = list(full_model.parameters())
t0 = time.perf_counter()
runs["full FT"] = (full_model, *train(full_model, params_full, steps=40, lr=1e-5), sum(p.numel() for p in params_full))
print(f"full FT : {time.perf_counter() - t0:.1f} 秒")

print(f"\n{'':10s} | {'学習パラメータ':>14} | {'Adam 状態':>10} | {'最終損失':>8} | {'平均 log p(正解)':>15} | {'KL(無関係な文)':>14}")
print(f"{'学習前':10s} | {'':>14} | {'':>10} | {'':>8} | {mean_target_logprob(base_model):>15.2f} | {0:>14.4f}")
for name, (m, losses, track, n_p) in runs.items():
    print(f"{name:10s} | {n_p:>14,} | {n_p * 8 / 1e6:>8.1f}MB | {losses[-1]:>8.3f} | {track[-1][1]:>15.2f} | {track[-1][2]:>14.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for name, (_, losses, track, _) in runs.items():
    axes[0].plot(range(1, len(losses) + 1), losses, label=name)
    _, lp, kl = zip(*track)
    axes[1].plot(kl, lp, "o-", label=name)
axes[0].set_xlabel("step"); axes[0].set_ylabel("training loss"); axes[0].set_yscale("log")
axes[0].set_title("training loss"); axes[0].legend()
axes[1].set_xscale("symlog", linthresh=0.01)
axes[1].set_xlabel("KL on unrelated text (forgetting)  →"); axes[1].set_ylabel("mean log p(target) (learning)  →")
axes[1].set_title("learning vs. forgetting (a point every 5 steps, step 0 at lower left)"); axes[1].legend()
plt.tight_layout(); plt.show()
for name in ["LoRA r=1", "full FT"]:
    report(runs[name][0], name)

**観察**（数値は実行環境によって多少変わります）：
- **パラメータとメモリ**：LoRA $r=8$ は全体の 0.24%、$r=1$ は 0.03% のパラメータしか動かしません。Adam の状態は、full fine-tuning の約 1 GB に対して LoRA $r=8$ は約 2 MB です。
- **ランク**：この課題（9 文の暗記）では、$r=1$ でも $r=8$ とほぼ同じ速さで学習できました。原論文でも、ごく小さな $r$ で十分な性能が出る場合が多いと報告されています。
- **忘却**：右図は、学習が進むにつれて「覚えた度合い」と「忘れた度合い」がどう動くかを示します（左上ほど「よく覚えて、あまり忘れていない」）。この小さな実験では、同じくらい覚えた時点で比べると、LoRA（`c_attn` だけに入れ、大きな学習率）の方が無関係な文章での KL が大きく、France の首都まで "Quexton" と答えるようになりました。full fine-tuning（小さな学習率）は "Paris" を保っています。ただし full fine-tuning は 40 ステップの時点では言い回しを変えたプロンプトをまだ十分に覚えておらず、覚えた量自体も少なめです。
- つまり **「LoRA だから元の知識を忘れない」とは限りません**。忘却の程度は学習率・ステップ数・データの偏り・LoRA を入れる場所などに強く依存します。Biderman et al. (2024, *LoRA Learns Less and Forgets Less*) は、より大規模な設定で「LoRA は full fine-tuning より学習量が少ないが忘却も少ない」と報告していますが、条件によって結果は変わります。実際の微調整では、学習率を下げる・早めに止める・一般的なデータを混ぜて学習する、といった工夫で過剰な一般化を抑えます。

In [ ]:
del runs, model_r1, full_model, params_full
if device == "cuda":
    torch.cuda.empty_cache()

## 5. メモリの見積もりと派生手法

### 学習時のメモリ（活性化を除く）

Adam で学習するとき、**学習するパラメータ** 1 個あたりに必要なメモリは、混合精度学習でよく使われる構成では

$$
\underbrace{2}_{\text{bf16 の重み}} + \underbrace{2}_{\text{bf16 の勾配}} + \underbrace{4}_{\text{fp32 のマスター重み}} + \underbrace{4 + 4}_{\text{Adam の } m, v} = 16\ \text{バイト}
$$

です。一方、**凍結したパラメータ** は重みを保持するだけ（bf16 なら 2 バイト）で済みます。LoRA はほとんどのパラメータを凍結するので、学習時のメモリが大きく減ります。さらに QLoRA は凍結した重みを 4 ビットに量子化します（約 0.5 バイト/パラメータ）。

In [ ]:
def gb(x):
    return f"{x / 1e9:8.2f} GB"


# 例: 7B 規模のモデル（32 層, d = 4096）で、各層の W_Q と W_V に r = 8 の LoRA
N = 7e9
n_lora = 32 * 2 * 8 * (4096 + 4096)
print(f"LoRA のパラメータ数: {n_lora:,}（全体の {n_lora / N:.3%}）\n")
print("7B モデルの学習時メモリ（重み・勾配・Adam 状態のみ。活性化は除く）")
print(f"  full fine-tuning（16 バイト/パラメータ）       : {gb(N * 16)}")
print(f"  LoRA（凍結部分 bf16 2 バイト + LoRA 16 バイト）: {gb(N * 2 + n_lora * 16)}")
print(f"  QLoRA（凍結部分 4 ビット ≈ 0.5 バイト + LoRA） : {gb(N * 0.5 + n_lora * 16)}")
print(f"\nチェックポイントのサイズ: 全重み bf16 {gb(N * 2)} / LoRA だけ bf16 {n_lora * 2 / 1e6:.1f} MB")

原論文では GPT-3 175B に対して、学習時の GPU メモリが 1.2 TB から 350 GB に減り、$r=4$ で $W_Q, W_V$ だけに LoRA を入れた場合のチェックポイントは 350 GB から 35 MB（約 1 万分の 1）になったと報告されています。

### どこに・どのランクで入れるか

原論文の GPT-3 での実験では、同じパラメータ数の予算なら $W_Q$ だけに大きなランクで入れるより、$W_Q$ と $W_V$ の両方に小さなランク（$r = 4$ 程度）で入れる方がよく、$r$ は 1〜8 程度の小さな値でも十分な性能が出たと報告されています。現在は、注意層の全射影（$W_Q, W_K, W_V, W_O$）や MLP 層にも入れる設定がよく使われます。

### 主な派生手法

| 手法 | アイデア |
|---|---|
| QLoRA (Dettmers et al., 2023) | 凍結したベースモデルを 4 ビット（NF4）に量子化し、その上で LoRA を学習。単一 GPU で大きなモデルを微調整できる |
| AdaLoRA (Zhang et al., 2023) | 層ごとに必要なランクが違うことに着目し、学習中にランクの配分を調整する |
| rsLoRA (Kalajdzievski, 2023) | スケーリングを $\alpha/r$ から $\alpha/\sqrt{r}$ に変え、大きな $r$ でも学習が遅くならないようにする |
| DoRA (Liu et al., 2024) | 重みを「大きさ」と「方向」に分解し、方向の更新に LoRA を使う |

## 6. まとめと参考文献

### このノートで確認したこと

| 節 | 確認したこと |
|---|---|
| 1 | パラメータ数 $r(d_{\text{in}}+d_{\text{out}})$、$B=0$ 初期化で出力が元のモデルと一致、$\mathrm{rank}(\Delta W)\le r$、$(xA)B$ の順で安く計算、マージしても出力不変 |
| 2 | $\partial\mathcal{L}/\partial A = s\nabla_W\mathcal{L}\,B^\top$、$\partial\mathcal{L}/\partial B = sA^\top\nabla_W\mathcal{L}$（数値微分で検証）。最初は $B$ だけが動く、$A=B=0$ は鞍点、1 ステップ目の更新は full FT の勾配の射影 |
| 3 | LoRA の到達損失 $=\mathcal{L}_{\min} + \sum_{i>r}\sigma_i^2$（Eckart–Young）が学習結果と一致。タスクの変化が低ランクなら full FT の更新も実質低ランク。LoRA が見つけた方向は真の方向と一致 |
| 4 | GPT-2 small に自作 LoRA（0.24% のパラメータ）で架空の知識を学習。ベースの重みは不変、初回の $A$ の勾配は 0、$\Delta W$ のランクは $r$、マージ・付け外し・保存（約 1 MB）、`peft` と一致。偏ったデータでは過剰な一般化（France → Quexton）が起き、忘却は LoRA でも起こりうる |
| 5 | 学習時メモリと保存サイズの見積もり、派生手法 |

### 参考文献
- E. J. Hu, Y. Shen, P. Wallis, Z. Allen-Zhu, Y. Li, S. Wang, L. Wang, W. Chen. "LoRA: Low-Rank Adaptation of Large Language Models." ICLR 2022. [arXiv:2106.09685](https://arxiv.org/abs/2106.09685) / [microsoft/LoRA](https://github.com/microsoft/LoRA)
- Hugging Face PEFT: [huggingface/peft](https://github.com/huggingface/peft)
- 本文中で触れた研究：Aghajanyan et al. (2020) *Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning*、Dettmers et al. (2023) *QLoRA: Efficient Finetuning of Quantized LLMs*、Zhang et al. (2023) *AdaLoRA*、Kalajdzievski (2023) *A Rank Stabilization Scaling Factor for Fine-Tuning with LoRA*、Liu et al. (2024) *DoRA: Weight-Decomposed Low-Rank Adaptation*、Biderman et al. (2024) *LoRA Learns Less and Forgets Less*